<a href="https://colab.research.google.com/github/DhikshaSubash/market-sentiment-analysis/blob/main/notebooks/03_spark_etl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 - Install dependencies
!pip install pyspark nltk -q

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

print("✅ Libraries ready!")

In [ ]:
# Cell 2 - Load raw data from Drive
from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MarketSentimentETL") \
    .getOrCreate()

# Load saved parquet files
news_df = spark.read.parquet("/content/drive/MyDrive/market_sentiment/raw/news")
price_df = spark.read.parquet("/content/drive/MyDrive/market_sentiment/raw/prices")

print(f"✅ News records loaded: {news_df.count()}")
print(f"✅ Price records loaded: {price_df.count()}")
news_df.show(3, truncate=50)

In [ ]:
# Cell 3 - Clean raw headlines
from pyspark.sql.functions import (
    lower, regexp_replace, trim, col
)

def clean_headlines(df):
    return df \
        .withColumn("cleaned", lower(col("headline"))) \
        .withColumn("cleaned", regexp_replace(col("cleaned"), r"http\S+", "")) \
        .withColumn("cleaned", regexp_replace(col("cleaned"), r"[^a-zA-Z\s]", "")) \
        .withColumn("cleaned", regexp_replace(col("cleaned"), r"\s+", " ")) \
        .withColumn("cleaned", trim(col("cleaned")))

news_clean_df = clean_headlines(news_df)

print("✅ Headlines cleaned!")
print("\nBefore vs After:")
news_clean_df.select("headline", "cleaned").show(3, truncate=60)

In [ ]:
# Cell 4 - Tokenization (split headline into words)
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(inputCol="cleaned", outputCol="tokens")
news_tokenized_df = tokenizer.transform(news_clean_df)

print("✅ Tokenization done!")
print("\nSample tokens:")
news_tokenized_df.select("cleaned", "tokens").show(3, truncate=60)

In [ ]:
# Cell 5 - Remove stop words (the, is, at, which, etc.)
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(inputCol="tokens", outputCol="filtered_tokens")
news_filtered_df = remover.transform(news_tokenized_df)

print("✅ Stop words removed!")
print("\nTokens vs Filtered:")
news_filtered_df.select("tokens", "filtered_tokens").show(3, truncate=60)

In [ ]:
# Cell 6 - Lemmatization (running → run, stocks → stock)
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_words(tokens):
    if tokens is None:
        return []
    return [lemmatizer.lemmatize(word) for word in tokens]

lemmatize_udf = udf(lemmatize_words, ArrayType(StringType()))

news_lemmatized_df = news_filtered_df.withColumn(
    "lemmatized_tokens",
    lemmatize_udf(col("filtered_tokens"))
)

print("✅ Lemmatization done!")
print("\nFiltered vs Lemmatized:")
news_lemmatized_df.select("filtered_tokens", "lemmatized_tokens").show(3, truncate=60)

In [ ]:
# Cell 7 - Standardize price data
from pyspark.sql.functions import round as spark_round, to_timestamp

price_clean_df = price_df \
    .withColumn("price", spark_round(col("price"), 2)) \
    .withColumn("timestamp", to_timestamp(col("timestamp"))) \
    .dropDuplicates(["symbol", "timestamp"]) \
    .orderBy("symbol", "timestamp")

print("✅ Price data cleaned!")
print(f"Price records after dedup: {price_clean_df.count()}")
price_clean_df.show(3)

In [ ]:
# Cell 8 - Save processed data
# Select final columns for news
news_final_df = news_lemmatized_df.select(
    "symbol",
    "headline",
    "cleaned",
    "lemmatized_tokens",
    "source",
    "published_at",
    "ingested_at"
)

# Save both
news_final_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/market_sentiment/processed/news"
)
price_clean_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/market_sentiment/processed/prices"
)

print("✅ Processed data saved!")
print(f"\nFinal news records: {news_final_df.count()}")
print(f"Final price records: {price_clean_df.count()}")
print("\nFinal news schema:")
news_final_df.printSchema()